In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import pyarrow.parquet as pq

In [ ]:
df_bicycle = pd.read_parquet("./output/bike_co2_3000_oulu.parquet")

In [ ]:
df_bicycle

In [ ]:
pois = pd.read_parquet("data/pois_per_hex_new_class_oulu.parquet")

In [ ]:
# Compute totals per category
category_totals = (
    pois.groupby("category", as_index=False)["count"]
    .sum()
    .rename(columns={"count": "total_city"})
)

print(category_totals)

In [ ]:
import h3
from shapely.geometry import Polygon

# Build geometry hexagonal a partir de h3_id
pois["geometry"] = pois["h3_id"].apply(
    lambda h: Polygon(h3.h3_to_geo_boundary(str(h), geo_json=True))
)

In [ ]:
jobs = gpd.read_parquet("data/job_distribution_from_census_oulu.parquet")

In [ ]:
jobs = jobs.reset_index()

In [ ]:
jobs_as_pois = jobs.rename(columns={"weighted_tyo": "count", "ID": "h3_id"})
jobs_as_pois["category"] = "jobs"
jobs_as_pois = jobs_as_pois[["h3_id", "category", "count", "geometry"]]

pois_clean = pois[["h3_id", "category", "count", "geometry"]]

In [ ]:
jobs_as_pois["count"].sum()

In [ ]:
# Ensure pois is a GeoDataFrame
if not isinstance(pois, gpd.GeoDataFrame):
    pois = gpd.GeoDataFrame(pois, geometry="geometry", crs="EPSG:4326")

In [ ]:
pois_with_jobs = gpd.GeoDataFrame(
    pd.concat([pois_clean, jobs_as_pois], ignore_index=True),
    crs=pois.crs
)

In [ ]:
df_bicycle.head()

In [ ]:
pois_with_jobs

In [ ]:
pois_grouped = (
    pois_with_jobs
    .pivot_table(
        index=["h3_id", "geometry"],
        columns="category",
        values="count",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

In [ ]:
pois_grouped

In [ ]:

df_bic_co2_merged = df_bicycle.merge(pois_grouped, how='left', left_on='to_id', right_on='h3_id')


# Exclude non-numeric columns (like h3_id and geometry)
category_cols = [
    col for col in pois_grouped.columns 
    if col not in ["h3_id", "geometry"]
]

# Fill only numeric category columns
for col in category_cols:
    df_bic_co2_merged[col] = df_bic_co2_merged[col].fillna(0).astype(int)


In [ ]:
df_bic_co2_merged.head()

In [ ]:
# Self merge to find reverse pairs
df_return_bic = df_bic_co2_merged.merge(
    df_bic_co2_merged[["from_id", "to_id", "co2_emissions_g"]],
    left_on=["from_id", "to_id"],
    right_on=["to_id", "from_id"],
    how="left",
    suffixes=("", "_return")
)

# Rename new column
df_return_bic = df_return_bic.rename(columns={"co2_emissions_g_return": "return_emission"})

# Fill NaN with original leg emission
df_return_bic["return_emission"] = df_return_bic["return_emission"].fillna(df_return_bic["co2_emissions_g"])

In [ ]:
# Step 1: keep outbound as separate column
df_return_bic = df_return_bic.rename(columns={"co2_emissions_g": "outbound_co2"})

# Step 2: replace pt_co2_total with return_emission
df_return_bic = df_return_bic.rename(columns={"return_emission": "return_co2"})

In [ ]:
df_return_bic["bike_co2_total"] = df_return_bic["outbound_co2"] + df_return_bic["return_co2"]

# Step 3: final DataFrame is the same variable name as before
df_bic_co2_merged = df_return_bic

In [ ]:
df_bic_co2_merged["bike_co2_total"] = df_bic_co2_merged["bike_co2_total"]/1000

In [ ]:
df_bic_co2_merged["total_poi"] = (
    df_bic_co2_merged["Education"]
    + df_bic_co2_merged["Healthcare and Health"]
    + df_bic_co2_merged["Recreational, Outdoors"]
    + df_bic_co2_merged["Shopping, Errands"]
    + df_bic_co2_merged["Social, Cultural"]
)

In [ ]:
# Known totals
total_jobs = 100459.0
total_pois = 5835

df_bic_co2_merged["jobs_pct"] = (df_bic_co2_merged["jobs"] / total_jobs) * 100
df_bic_co2_merged["pois_pct"] = (df_bic_co2_merged["total_poi"] / total_pois) * 100

df_bic_co2_merged["total_pct"] = (
    df_bic_co2_merged[["jobs_pct", "pois_pct"]].mean(axis=1)
)

In [ ]:
df_bic_co2_merged.columns

In [ ]:
# Filter rows where pt_time is less than or equal to 15, 30, and 45
df_bike_co2_1000 = df_bic_co2_merged[df_bic_co2_merged["bike_co2_total"] <= 1].copy()
df_bike_co2_500 = df_bic_co2_merged[df_bic_co2_merged["bike_co2_total"] <= 0.5].copy()

In [ ]:
df_bike_co2_1000

In [ ]:
df_bike_co2_1000.columns

In [ ]:
# --- Step 0: totals in the whole city
totals = {
    "Education": 292,
    "Healthcare and Health": 361,
    "Others / Not sure": 1486,
    "Recreational, Outdoors": 1167,
    "Shopping, Errands": 1404,
    "Social, Cultural": 1125,
    "jobs": 100459
}

# --- Step 1: categories to use
cols = list(totals.keys())

# --- Step 2: aggregate counts per from_id
df_grouped = (
    df_bike_co2_1000.groupby("from_id", as_index=False)[cols]
    .sum()
)

# --- Step 3: compute percentages relative to total city values
for col in cols:
    df_grouped[col + "_pct"] = (df_grouped[col] / totals[col]) * 100

# --- Optional Step 4: merge percentages back into the trip dataframe
df_bike_co2_1000 = df_bike_co2_1000.merge(
    df_grouped[["from_id"] + [c + "_pct" for c in cols]],
    on="from_id",
    how="left"
)

df_grouped.head()




In [ ]:
# Step: compute total_pois for each from_id
df_grouped["total_pois"] = df_grouped[[
    "Education",
    "Healthcare and Health",
    "Recreational, Outdoors",
    "Shopping, Errands",
    "Social, Cultural",
    "jobs"
]].sum(axis=1)

# Step: compute percentage relative to city total (same categories only)
city_total_pois = (
    totals["Education"]
    + totals["Healthcare and Health"]
    + totals["Recreational, Outdoors"]
    + totals["Shopping, Errands"]
    + totals["Social, Cultural"]
    + totals["jobs"]
)

df_grouped["total_pois_pct"] = (df_grouped["total_pois"] / city_total_pois) * 100


In [ ]:
df_grouped["mean_pois_jobs_pct"] = df_grouped[["total_pois_pct", "jobs_pct"]].mean(axis=1)

In [ ]:
df_grouped.to_parquet("./output/bike_map_data_overall_oulu.parquet")

In [ ]:
df_grouped

In [ ]:
import h3
from shapely.geometry import Polygon
import contextily as ctx; import basemaps
import matplotlib.pyplot as plt
import mapclassify
import matplotlib.patches as mpatches
import geopandas as gpd

# Function to convert h3 to polygon
def h3_to_polygon(h):
    return Polygon(h3.h3_to_geo_boundary(h, geo_json=True))

# Convert 'from_id' to geometry
grouped_summary = df_grouped.copy()
geometry = grouped_summary['from_id'].apply(h3_to_polygon)
gdf_hexes = gpd.GeoDataFrame(grouped_summary, geometry=geometry, crs='EPSG:4326')

# Filter hexagons with mean_pois_jobs_pct >= 0.5%
gdf_hexes = gdf_hexes[gdf_hexes['total_pois_pct'] >= 0.5]

# Classify total_pois into 5 natural breaks
classifier = mapclassify.NaturalBreaks(y=gdf_hexes['total_pois_pct'], k=5)
gdf_hexes['poi_class'] = classifier.yb

# Get bin edges for legend
bin_edges = classifier.bins

# Define custom legend handles with % sign
legend_handles = []
for i in range(len(bin_edges)):
    if i == 0:
        label = f"<= {bin_edges[i]:.1f}%"
    else:
        label = f"> {bin_edges[i-1]:.1f}% – {bin_edges[i]:.1f}%"
    patch = mpatches.Patch(color=plt.cm.viridis(i / (len(bin_edges)-1)), label=label)
    legend_handles.append(patch)

# Plot
fig, ax = plt.subplots(figsize=(10, 10))
gdf_hexes.plot(ax=ax, column='poi_class', cmap='viridis', legend=False, edgecolor='black', linewidth=0.2)
ctx.add_basemap(ax, source=basemaps.POSITRON, crs=gdf_hexes.crs.to_string())
ax.set_title("Accessible POIs (% of city total) by cycling with 1 kg CO₂ budget")
ax.axis('off')
plt.legend(handles=legend_handles, title="Share of POIs and Jobs (%)", loc='lower left')
plt.tight_layout()
plt.show()


```python
import h3
from shapely.geometry import Polygon
import geopandas as gpd
import contextily as ctx; import basemaps
import matplotlib.pyplot as plt
import mapclassify
import matplotlib.patches as mpatches

# Function to convert h3 to polygon
def h3_to_polygon(h):
    return Polygon(h3.h3_to_geo_boundary(h, geo_json=True))

# Define POI percentage columns to plot
poi_categories = [
    'Education_pct',
    'Healthcare and Health_pct',
    'Recreational, Outdoors_pct',
    'Shopping, Errands_pct',
    'Social, Cultural_pct',
    'jobs_pct'
]

# Define nice display names for categories
category_labels = {
    'Education_pct': "Education",
    'Healthcare and Health_pct': "Healthcare",
    'Recreational, Outdoors_pct': "Recreation & Outdoors",
    'Shopping, Errands_pct': "Shopping & Errands",
    'Social, Cultural_pct': "Social & Cultural",
    'jobs_pct': "Jobs"
}

# Create GeoDataFrame with geometry if not already created
if 'geometry' not in df_grouped.columns:
    geometry = df_grouped['from_id'].apply(h3_to_polygon)
    gdf_hexes = gpd.GeoDataFrame(df_grouped, geometry=geometry, crs='EPSG:4326')
else:
    gdf_hexes = df_grouped.copy()

# Filter out very low percentages (<0.5%)
gdf_hexes = gdf_hexes[gdf_hexes[poi_categories].sum(axis=1) >= 0.5]

# Set up plot grid
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

# Generate map for each category
for idx, category in enumerate(poi_categories):
    ax = axes[idx]
    
    # Classify using Natural Breaks
    classifier = mapclassify.NaturalBreaks(y=gdf_hexes[category], k=5)
    gdf_hexes['poi_class'] = classifier.yb
    bin_edges = classifier.bins

    # Create custom legend with %
    legend_handles = []
    for i in range(len(bin_edges)):
        if i == 0:
            label = f"≤ {bin_edges[i]:.0f}%"
        else:
            label = f"{bin_edges[i-1]:.0f}% – {bin_edges[i]:.0f}%"
        patch = mpatches.Patch(
            color=plt.cm.viridis(i / (len(bin_edges) - 1)),
            label=label
        )
        legend_handles.append(patch)

    # Plot hexagons
    gdf_hexes.plot(
        ax=ax, column='poi_class', cmap='viridis',
        legend=False, edgecolor='black', linewidth=0.2
    )
    ctx.add_basemap(ax, source=basemaps.POSITRON, crs=gdf_hexes.crs.to_string())
    ax.set_title(category_labels[category])
    ax.axis('off')
    ax.legend(handles=legend_handles, title=category_labels[category], loc='lower left')

# Add main title
plt.suptitle(
    "Accessible POIs (% of city total) by cycling with 1 kg CO₂ budget",
    fontsize=20
)

# Adjust spacing to prevent overlap
plt.tight_layout(rect=[0, 0.02, 1, 0.98])
plt.subplots_adjust(hspace=0.15, wspace=0.15)
plt.show()
```

### Public Transport

In [ ]:
df_pt_co2 = pd.read_parquet("./output/pt_co2_3000_oulu.parquet")


In [ ]:

df_pt_co2_merged = df_pt_co2.merge(pois_grouped, how='left', left_on='to_id', right_on='h3_id')


# Exclude non-numeric columns (like h3_id and geometry)
category_cols = [
    col for col in pois_grouped.columns 
    if col not in ["h3_id", "geometry"]
]

# Fill only numeric category columns
for col in category_cols:
    df_pt_co2_merged[col] = df_pt_co2_merged[col].fillna(0).astype(int)

In [ ]:
df_pt_co2_merged

In [ ]:
# Self merge to find reverse pairs
df_return_pt = df_pt_co2_merged.merge(
    df_pt_co2_merged[["from_id", "to_id", "pt_co2_total"]],
    left_on=["from_id", "to_id"],
    right_on=["to_id", "from_id"],
    how="left",
    suffixes=("", "_return")
)

# Rename new column
df_return_pt = df_return_pt.rename(columns={"pt_co2_total_return": "return_emission"})

# Fill NaN with original leg emission
df_return_pt["return_emission"] = df_return_pt["return_emission"].fillna(df_return_pt["pt_co2_total"])

In [ ]:
# Step 1: keep outbound as separate column
df_return_pt = df_return_pt.rename(columns={"pt_co2_total": "outbound_co2"})

# Step 2: replace pt_co2_total with return_emission
df_return_pt = df_return_pt.rename(columns={"return_emission": "return_co2"})

In [ ]:
df_return_pt["pt_co2_total"] = df_return_pt["outbound_co2"] + df_return_pt["return_co2"]

# Step 3: final DataFrame is the same variable name as before
df_pt_co2_merged = df_return_pt

In [ ]:
df_pt_co2_merged["total_poi"] = (
    df_pt_co2_merged["Education"]
    + df_pt_co2_merged["Healthcare and Health"]
    + df_pt_co2_merged["Recreational, Outdoors"]
    + df_pt_co2_merged["Shopping, Errands"]
    + df_pt_co2_merged["Social, Cultural"]
)

In [ ]:
# Known totals
total_jobs = 100459.0
total_pois = 5835

df_pt_co2_merged["jobs_pct"] = (df_pt_co2_merged["jobs"] / total_jobs) * 100
df_pt_co2_merged["pois_pct"] = (df_pt_co2_merged["total_poi"] / total_pois) * 100

df_pt_co2_merged["total_pct"] = (
    df_pt_co2_merged[["jobs_pct", "pois_pct"]].mean(axis=1)
)

In [ ]:
# Filter rows where pt_time is less than or equal to 15, 30, and 45
df_pt_co2_1000 = df_pt_co2_merged[df_pt_co2_merged["pt_co2_total"] <= 1000].copy()
df_pt_co2_500 = df_pt_co2_merged[df_pt_co2_merged["pt_co2_total"] <= 500].copy()

In [ ]:
df_pt_co2_1000.columns

In [ ]:
# --- Step 0: totals in the whole city
totals = {
    "Education": 292,
    "Healthcare and Health": 361,
    "Others / Not sure": 1486,
    "Recreational, Outdoors": 1167,
    "Shopping, Errands": 1404,
    "Social, Cultural": 1125,
    "jobs": 100459
}

# --- Step 1: categories to use
cols = list(totals.keys())

# --- Step 2: aggregate counts per from_id
df_grouped = (
    df_pt_co2_1000.groupby("from_id", as_index=False)[cols]
    .sum()
)

# --- Step 3: compute percentages relative to total city values
for col in cols:
    df_grouped[col + "_pct"] = (df_grouped[col] / totals[col]) * 100

# --- Optional Step 4: merge percentages back into the trip dataframe
df_pt_co2_1000 = df_pt_co2_1000.merge(
    df_grouped[["from_id"] + [c + "_pct" for c in cols]],
    on="from_id",
    how="left"
)



In [ ]:
# Step: compute total_pois for each from_id
df_grouped["total_pois"] = df_grouped[[
    "Education",
    "Healthcare and Health",
    "Recreational, Outdoors",
    "Shopping, Errands",
    "Social, Cultural",
    "jobs"
]].sum(axis=1)

# Step: compute percentage relative to city total (same categories only)
city_total_pois = (
    totals["Education"]
    + totals["Healthcare and Health"]
    + totals["Recreational, Outdoors"]
    + totals["Shopping, Errands"]
    + totals["Social, Cultural"]
    + totals["jobs"]
)

df_grouped["total_pois_pct"] = (df_grouped["total_pois"] / city_total_pois) * 100


In [ ]:
df_grouped["mean_pois_jobs_pct"] = df_grouped[["total_pois_pct", "jobs_pct"]].mean(axis=1)

In [ ]:
df_grouped.to_parquet("./output/pt_map_data_overall_oulu.parquet")

In [ ]:
df_grouped

In [ ]:
import h3
from shapely.geometry import Polygon
import contextily as ctx; import basemaps
import matplotlib.pyplot as plt
import mapclassify
import matplotlib.patches as mpatches
import geopandas as gpd

# Function to convert h3 to polygon
def h3_to_polygon(h):
    return Polygon(h3.h3_to_geo_boundary(h, geo_json=True))

# Convert 'from_id' to geometry
grouped_summary = df_grouped.copy()
geometry = grouped_summary['from_id'].apply(h3_to_polygon)
gdf_hexes = gpd.GeoDataFrame(grouped_summary, geometry=geometry, crs='EPSG:4326')

# Filter hexagons with mean_pois_jobs_pct >= 0.5%
gdf_hexes = gdf_hexes[gdf_hexes['mean_pois_jobs_pct'] >= 0.5]

# Classify total_pois into 5 natural breaks
classifier = mapclassify.NaturalBreaks(y=gdf_hexes['mean_pois_jobs_pct'], k=5)
gdf_hexes['poi_class'] = classifier.yb

# Get bin edges for legend
bin_edges = classifier.bins

# Define custom legend handles with % sign
legend_handles = []
for i in range(len(bin_edges)):
    if i == 0:
        label = f"<= {bin_edges[i]:.1f}%"
    else:
        label = f"> {bin_edges[i-1]:.1f}% – {bin_edges[i]:.1f}%"
    patch = mpatches.Patch(color=plt.cm.viridis(i / (len(bin_edges)-1)), label=label)
    legend_handles.append(patch)

# Plot
fig, ax = plt.subplots(figsize=(10, 10))
gdf_hexes.plot(ax=ax, column='poi_class', cmap='viridis', legend=False, edgecolor='black', linewidth=0.2)
ctx.add_basemap(ax, source=basemaps.POSITRON, crs=gdf_hexes.crs.to_string())
ax.set_title("Accessible POIs (% of city total) by Public Transport  with 1 kg CO₂ budget")
ax.axis('off')
plt.legend(handles=legend_handles, title="Share of POIs (%)", loc='lower left')
plt.tight_layout()
plt.show()

#fig.savefig("./output/Acc_PT_1kg_map.png", dpi=300, bbox_inches="tight")


```python
# Function to convert h3 to polygon
def h3_to_polygon(h):
    return Polygon(h3.h3_to_geo_boundary(h, geo_json=True))

# Define POI percentage columns to plot
poi_categories = [
    'Education_pct',
    'Healthcare and Health_pct',
    'Recreational, Outdoors_pct',
    'Shopping, Errands_pct',
    'Social, Cultural_pct',
    'jobs_pct'
]

# Create GeoDataFrame with geometry if not already created
if 'geometry' not in df_grouped.columns:
    geometry = df_grouped['from_id'].apply(h3_to_polygon)
    gdf_hexes = gpd.GeoDataFrame(df_grouped, geometry=geometry, crs='EPSG:4326')
else:
    gdf_hexes = df_grouped.copy()

# Filter out very low percentages (<0.5%)
gdf_hexes = gdf_hexes[gdf_hexes[poi_categories].sum(axis=1) >= 0.5]

# Set up plot grid
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

# Generate map for each category
for idx, category in enumerate(poi_categories):
    ax = axes[idx]
    
    # Classify using Natural Breaks
    classifier = mapclassify.NaturalBreaks(y=gdf_hexes[category], k=5)
    gdf_hexes['poi_class'] = classifier.yb
    bin_edges = classifier.bins

    # Create custom legend with %
    legend_handles = []
    for i in range(len(bin_edges)):
        if i == 0:
            label = f"<= {bin_edges[i]:.1f}%"
        else:
            label = f"> {bin_edges[i-1]:.1f}% – {bin_edges[i]:.1f}%"
        patch = mpatches.Patch(color=plt.cm.viridis(i / (len(bin_edges) - 1)), label=label)
        legend_handles.append(patch)

    # Plot hexagons
    gdf_hexes.plot(ax=ax, column='poi_class', cmap='viridis', legend=False, edgecolor='black', linewidth=0.2)
    ctx.add_basemap(ax, source=basemaps.POSITRON, crs=gdf_hexes.crs.to_string())
    ax.set_title(category.replace("_pct",""))
    ax.axis('off')
    ax.legend(handles=legend_handles, title=category.replace("_pct",""), loc='lower left')

plt.suptitle(
    "Accessible POIs (% of city total) by Public Transport  with 1 kg CO₂ budget",
    fontsize=20
)
# Adjust spacing to prevent legend overlap
plt.tight_layout(rect=[0, 0.02, 1, 0.98])  # leave extra space at the bottom
plt.subplots_adjust(hspace=0.1, wspace=0.1)  # add spacing between rows and columns
plt.show()
```

### Cars

In [ ]:
df_car_co2 = pd.read_parquet("scratch/car_co2_6000_oulu.parquet")

In [ ]:
df_car_co2.head()

In [ ]:
df_car_co2 = df_car_co2.rename(
    columns={
        "Origin_Hexagon_ID": "from_id",
        "Destination_Hexagon_ID": "to_id"
    }
)

In [ ]:

df_car_co2_merged = df_car_co2.merge(pois_grouped, how='left', left_on='to_id', right_on='h3_id')


# Exclude non-numeric columns (like h3_id and geometry)
category_cols = [
    col for col in pois_grouped.columns 
    if col not in ["h3_id", "geometry"]
]

# Fill only numeric category columns
for col in category_cols:
    df_car_co2_merged[col] = df_car_co2_merged[col].fillna(0).astype(int)

In [ ]:
# Self merge to find reverse pairs
df_return_car = df_car_co2_merged.merge(
    df_car_co2_merged[["from_id", "to_id", "car_co2"]],
    left_on=["from_id", "to_id"],
    right_on=["to_id", "from_id"],
    how="left",
    suffixes=("", "_return")
)

# Rename new column
df_return_car = df_return_car.rename(columns={"car_co2_return": "return_emission"})

# Fill NaN with original leg emission
df_return_car["return_emission"] = df_return_car["return_emission"].fillna(df_return_car["car_co2"])

In [ ]:
# Step 1: keep outbound as separate column
df_return_car = df_return_car.rename(columns={"car_co2": "outbound_co2"})

# Step 2: replace pt_co2_total with return_emission
df_return_car = df_return_car.rename(columns={"return_emission": "return_co2"})

In [ ]:
df_return_car["car_co2_total"] = df_return_car["outbound_co2"] + df_return_car["return_co2"]

# Step 3: final DataFrame is the same variable name as before
df_car_co2_merged = df_return_car

In [ ]:
df_car_co2_merged["total_poi"] = (
    df_car_co2_merged["Education"]
    + df_car_co2_merged["Healthcare and Health"]
    + df_car_co2_merged["Recreational, Outdoors"]
    + df_car_co2_merged["Shopping, Errands"]
    + df_car_co2_merged["Social, Cultural"]
)

In [ ]:
# Known totals
total_jobs = 100459.0
total_pois = 5835

df_pt_co2_merged["jobs_pct"] = (df_pt_co2_merged["jobs"] / total_jobs) * 100
df_pt_co2_merged["pois_pct"] = (df_pt_co2_merged["total_poi"] / total_pois) * 100

df_pt_co2_merged["total_pct"] = (
    df_pt_co2_merged[["jobs_pct", "pois_pct"]].mean(axis=1)
)

In [ ]:
# Filter rows where pt_time is less than or equal to 15, 30, and 45
df_car_co2_1000 = df_car_co2_merged[df_car_co2_merged["car_co2_total"] <= 1000].copy()
df_car_co2_500 = df_car_co2_merged[df_car_co2_merged["car_co2_total"] <= 500].copy()

In [ ]:
# --- Step 0: totals in the whole city
totals = {
    "Education": 292,
    "Healthcare and Health": 361,
    "Others / Not sure": 1486,
    "Recreational, Outdoors": 1167,
    "Shopping, Errands": 1404,
    "Social, Cultural": 1125,
    "jobs": 100459
}

# --- Step 1: categories to use
cols = list(totals.keys())

# --- Step 2: aggregate counts per from_id
df_grouped = (
    df_car_co2_1000.groupby("from_id", as_index=False)[cols]
    .sum()
)

# --- Step 3: compute percentages relative to total city values
for col in cols:
    df_grouped[col + "_pct"] = (df_grouped[col] / totals[col]) * 100

# --- Optional Step 4: merge percentages back into the trip dataframe
df_car_co2_1000 = df_car_co2_1000.merge(
    df_grouped[["from_id"] + [c + "_pct" for c in cols]],
    on="from_id",
    how="left"
)


In [ ]:
# Step: compute total_pois for each from_id
df_grouped["total_pois"] = df_grouped[[
    "Education",
    "Healthcare and Health",
    "Recreational, Outdoors",
    "Shopping, Errands",
    "Social, Cultural",
    "jobs"
]].sum(axis=1)

# Step: compute percentage relative to city total (same categories only)
city_total_pois = (
    totals["Education"]
    + totals["Healthcare and Health"]
    + totals["Recreational, Outdoors"]
    + totals["Shopping, Errands"]
    + totals["Social, Cultural"]
    + totals["jobs"]
)

df_grouped["total_pois_pct"] = (df_grouped["total_pois"] / city_total_pois) * 100

In [ ]:
df_grouped["mean_pois_jobs_pct"] = df_grouped[["total_pois_pct", "jobs_pct"]].mean(axis=1)

In [ ]:
df_grouped.to_parquet("./output/car_map_data_overall_oulu.parquet")

In [ ]:
from shapely.geometry import Polygon
import contextily as ctx; import basemaps
import matplotlib.pyplot as plt
import mapclassify
import matplotlib.patches as mpatches

# Function to convert h3 to polygon
def h3_to_polygon(h):
    return Polygon(h3.h3_to_geo_boundary(h, geo_json=True))

# Convert 'from_id' to geometry
grouped_summary = df_grouped.copy()
geometry = grouped_summary['from_id'].apply(h3_to_polygon)
gdf_hexes = gpd.GeoDataFrame(grouped_summary, geometry=geometry, crs='EPSG:4326')

# Filter hexagons with mean_pois_jobs_pct >= 0.5%
gdf_hexes = gdf_hexes[gdf_hexes['mean_pois_jobs_pct'] >= 0.05]

# Classify total_pois into 5 natural breaks
classifier = mapclassify.NaturalBreaks(y=gdf_hexes['mean_pois_jobs_pct'], k=5)
gdf_hexes['poi_class'] = classifier.yb

# Get bin edges for legend
bin_edges = classifier.bins

# Define custom legend handles with % sign
legend_handles = []
for i in range(len(bin_edges)):
    if i == 0:
        label = f"<= {bin_edges[i]:.1f}%"
    else:
        label = f"> {bin_edges[i-1]:.1f}% – {bin_edges[i]:.1f}%"
    patch = mpatches.Patch(color=plt.cm.viridis(i / (len(bin_edges)-1)), label=label)
    legend_handles.append(patch)

# Plot
fig, ax = plt.subplots(figsize=(10, 10))
gdf_hexes.plot(ax=ax, column='poi_class', cmap='viridis', legend=False, edgecolor='black', linewidth=0.2)
ctx.add_basemap(ax, source=basemaps.POSITRON, crs=gdf_hexes.crs.to_string())
ax.set_title("Accessible POIs (% of city total) by Car  with 1 kg CO₂ budget")
ax.axis('off')
plt.legend(handles=legend_handles, title="Share of POIs (%)", loc='lower left')
plt.tight_layout()
plt.show()


In [ ]:
# Function to convert h3 to polygon
def h3_to_polygon(h):
    return Polygon(h3.h3_to_geo_boundary(h, geo_json=True))

# Define POI percentage columns to plot
poi_categories = [
    'Education_pct',
    'Healthcare and Health_pct',
    'Recreational, Outdoors_pct',
    'Shopping, Errands_pct',
    'Social, Cultural_pct',
    'jobs_pct'
]

# Create GeoDataFrame with geometry if not already created
if 'geometry' not in df_grouped.columns:
    geometry = df_grouped['from_id'].apply(h3_to_polygon)
    gdf_hexes = gpd.GeoDataFrame(df_grouped, geometry=geometry, crs='EPSG:4326')
else:
    gdf_hexes = df_grouped.copy()

# Filter out very low percentages (<0.5%)
gdf_hexes = gdf_hexes[gdf_hexes[poi_categories].sum(axis=1) >= 0.5]

# Set up plot grid
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

# Generate map for each category
for idx, category in enumerate(poi_categories):
    ax = axes[idx]
    
    # Classify using Natural Breaks
    classifier = mapclassify.NaturalBreaks(y=gdf_hexes[category], k=5)
    gdf_hexes['poi_class'] = classifier.yb
    bin_edges = classifier.bins

    # Create custom legend with %
    legend_handles = []
    for i in range(len(bin_edges)):
        if i == 0:
            label = f"<= {bin_edges[i]:.1f}%"
        else:
            label = f"> {bin_edges[i-1]:.1f}% – {bin_edges[i]:.1f}%"
        patch = mpatches.Patch(color=plt.cm.viridis(i / (len(bin_edges) - 1)), label=label)
        legend_handles.append(patch)

    # Plot hexagons
    gdf_hexes.plot(ax=ax, column='poi_class', cmap='viridis', legend=False, edgecolor='black', linewidth=0.2)
    ctx.add_basemap(ax, source=basemaps.POSITRON, crs=gdf_hexes.crs.to_string())
    ax.set_title(category.replace("_pct",""))
    ax.axis('off')
    ax.legend(handles=legend_handles, title=category.replace("_pct",""), loc='lower left')

plt.suptitle(
    "Accessible POIs (% of city total) by Car  with 1 kg CO₂ budget",
    fontsize=20
)
# Adjust spacing to prevent legend overlap
plt.tight_layout(rect=[0, 0.02, 1, 0.98])  # leave extra space at the bottom
plt.subplots_adjust(hspace=0.1, wspace=0.1)  # add spacing between rows and columns
plt.show()

In [ ]:
import pandas as pd
cars = pd.read_parquet("./output/car_map_data_overall_oulu.parquet")
cars["Mode"] = "Car"

pt = pd.read_parquet("./output/pt_map_data_overall_oulu.parquet")
pt["Mode"] = "Public Transport"

bike = pd.read_parquet("./output/bike_map_data_overall_oulu.parquet")
bike["Mode"] = "Bike"


# Combine into a single DataFrame
df_all = pd.concat([bike, cars, pt], ignore_index=True)

# Quick check
df_all.head()

In [ ]:
import h3
from shapely.geometry import Polygon
import contextily as ctx; import basemaps
import matplotlib.pyplot as plt
import mapclassify
import matplotlib.patches as mpatches
import geopandas as gpd
# --- Convert from_id to geometry if missing ---
def h3_to_polygon(h):
    return Polygon(h3.h3_to_geo_boundary(h, geo_json=True))

if 'geometry' not in df_all.columns or df_all['geometry'].dtype == object:
    df_all["geometry"] = df_all["from_id"].apply(h3_to_polygon)

gdf_all = gpd.GeoDataFrame(df_all, geometry='geometry', crs='EPSG:4326')

# --- Filter very low values ---
gdf_all = gdf_all[gdf_all['mean_pois_jobs_pct'] >= 0.05]

# --- Custom bins and colors ---
bins = [0, 5, 15, 30, 50, 70, 85, 100]
colors = ["#ffffcc", "#c2e699", "#78c679", "#41ab5d", "#238443", "#005a32", "#004529"]  # light → dark green

# Assign class to each hexagon for all modes
gdf_all['poi_class'] = pd.cut(
    gdf_all['mean_pois_jobs_pct'],
    bins=bins,
    labels=range(len(bins)-1),
    include_lowest=True
)

# --- Plot grid: 1 row x 3 modes ---
mode_order = ["Bike", "Public Transport", "Car"]
fig, axes = plt.subplots(1, len(mode_order), figsize=(20, 8), sharex=True, sharey=True)

for ax, mode in zip(axes, mode_order):
    gdf_mode = gdf_all[gdf_all['Mode'] == mode].copy()
    
    if gdf_mode.empty:
        ax.set_title(mode, fontsize=16)
        ax.axis('off')
        continue

    gdf_mode.plot(
        ax=ax,
        color=[colors[int(x)] for x in gdf_mode['poi_class']],
        edgecolor='black',
        linewidth=0.2
    )
    ctx.add_basemap(ax, source=basemaps.POSITRON, crs=gdf_all.crs.to_string())
    ax.set_title(mode, fontsize=16)
    ax.axis('off')

# --- Legend ---
legend_labels = ["0–5%", "5–15%", "15–30%", "30–50%", "50–70%", "70–85%", ">85%"]
legend_handles = [mpatches.Patch(color=colors[i], label=legend_labels[i]) for i in range(len(legend_labels))]

fig.legend(
    handles=legend_handles,
    title="Share of POIs (%)",
    loc='lower center',
    ncol=len(legend_labels),
    fontsize=12,
    title_fontsize=14
)

# --- Title ---
plt.suptitle("Accessible POIs (% of city total) by mode with 1 kg CO₂ budget", fontsize=19)

plt.tight_layout(rect=[0, 0.08, 1, 0.99])
plt.show()

fig.savefig("./output/accessible_POIs_by_mode_1kg_CO2_oulu.png", dpi=300, bbox_inches='tight')



In [ ]:
import h3
from shapely.geometry import Polygon
import contextily as ctx; import basemaps
import matplotlib.pyplot as plt
import mapclassify
import matplotlib.patches as mpatches
import geopandas as gpd
import pandas as pd

# -------------------------------------------------
# H3 → geometry
# -------------------------------------------------
def h3_to_polygon(h):
    return Polygon(h3.h3_to_geo_boundary(h, geo_json=True))

# -------------------------------------------------
# CREATE GEOMETRY
# -------------------------------------------------
if 'geometry' not in df_all.columns or df_all['geometry'].dtype == object:

    df_all["geometry"] = df_all["from_id"].apply(h3_to_polygon)

gdf_all = gpd.GeoDataFrame(
    df_all,
    geometry='geometry',
    crs='EPSG:4326'
)

# -------------------------------------------------
# PROJECT CRS
# -------------------------------------------------
gdf_all = gdf_all.to_crs(epsg=3857)

# -------------------------------------------------
# LOAD FILTER HEXAGONS (TAMPERE)
# -------------------------------------------------
filter_hex = gpd.read_parquet(
    "./data/filter_hex_oulu.parquet"
).to_crs(epsg=3857)

# -------------------------------------------------
# KEEP ONLY HEXAGONS INSIDE FILTER
# -------------------------------------------------
gdf_all = gpd.sjoin(
    gdf_all,
    filter_hex[["geometry"]],
    how="inner",
    predicate="intersects"
)

gdf_all = gdf_all.drop(
    columns=["index_right"],
    errors="ignore"
)

# -------------------------------------------------
# FILTER VERY LOW VALUES
# -------------------------------------------------
gdf_all = gdf_all[
    gdf_all['mean_pois_jobs_pct'] >= 0.05
].copy()

# -------------------------------------------------
# VISUAL TRICK:
# ADD PT HEXAGONS MISSING IN CAR
# -------------------------------------------------
pt_hex = set(
    gdf_all[
        gdf_all["Mode"] == "Public Transport"
    ]["from_id"]
)

car_hex = set(
    gdf_all[
        gdf_all["Mode"] == "Car"
    ]["from_id"]
)

missing_hex = list(pt_hex - car_hex)

missing_rows = gdf_all[
    (gdf_all["Mode"] == "Public Transport") &
    (gdf_all["from_id"].isin(missing_hex))
].copy()

# assign almost-zero accessibility
missing_rows["mean_pois_jobs_pct"] = 0.001
missing_rows["Mode"] = "Car"

# append back
gdf_all = pd.concat(
    [
        gdf_all,
        missing_rows
    ],
    ignore_index=True
)

# -------------------------------------------------
# CUSTOM BINS + COLORS
# -------------------------------------------------
bins = [0, 5, 15, 30, 50, 70, 85, 100]

colors = [
    "#D9D9D9",   # neutral gray - lowest accessibility
    "#F0B8CE",
    "#E888B2",
    "#DB4894",   # matches C1
    "#B82E82",
    "#8C2277",   # deepens toward magenta-purple
    "#56007B",   # deepest - highest accessibility (matches C3)
]

# -------------------------------------------------
# CLASSIFY
# -------------------------------------------------
gdf_all['poi_class'] = pd.cut(
    gdf_all['mean_pois_jobs_pct'],
    bins=bins,
    labels=range(len(bins)-1),
    include_lowest=True
)

# -------------------------------------------------
# PLOT
# -------------------------------------------------
mode_order = [
    "Bike",
    "Public Transport",
    "Car"
]

fig, axes = plt.subplots(
    1,
    len(mode_order),
    figsize=(20, 8),
    sharex=True,
    sharey=True
)

for ax, mode in zip(axes, mode_order):

    gdf_mode = gdf_all[
        gdf_all['Mode'] == mode
    ].copy()

    if gdf_mode.empty:

        ax.set_title(mode, fontsize=16)
        ax.axis('off')
        continue

    # ---------------------------------------------
    # plot hexagons
    # ---------------------------------------------
    gdf_mode.plot(
        ax=ax,
        color=[colors[int(x)] for x in gdf_mode['poi_class']],
        edgecolor='black',
        linewidth=0.2
    )

    # ---------------------------------------------
    # basemap
    # ---------------------------------------------
    ctx.add_basemap(
        ax,
        source=basemaps.POSITRON,
        zoom=11
    )

    ax.set_title(mode, fontsize=16)
    ax.axis('off')

# -------------------------------------------------
# LEGEND
# -------------------------------------------------
legend_labels = [
    "0–5%",
    "5–15%",
    "15–30%",
    "30–50%",
    "50–70%",
    "70–85%",
    ">85%"
]

legend_handles = [

    mpatches.Patch(
        color=colors[i],
        label=legend_labels[i]
    )

    for i in range(len(legend_labels))
]

fig.legend(
    handles=legend_handles,
    title="Share of POIs (%)",
    loc='lower center',
    ncol=len(legend_labels),
    fontsize=12,
    title_fontsize=14
)

# -------------------------------------------------
# TITLE
# -------------------------------------------------
plt.suptitle(
    "Accessible POIs (% of city total) by mode with 1 kg CO₂ budget",
    fontsize=19
)

plt.tight_layout(rect=[0, 0.08, 1, 0.99])

# -------------------------------------------------
# SAVE
# -------------------------------------------------
fig.savefig(
    "./output/accessible_POIs_by_mode_1kg_CO2_oulu_filtered.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:

plt.figure(figsize=(10,7))

for origin, g in df_curves.groupby("Origin_Name"):
    plt.plot(
        g["car_co2_total"], 
        g["cumulative_pct"], 
        marker="o", 
        linewidth=1.2,   # thinner line
        markersize=4,    # smaller markers
        label=origin
    )

plt.xlabel("PT CO₂ emissions (grams)")
plt.ylabel("Cumulative opportunities reached (%)")
plt.title("Accessibility vs. Car CO₂ Emissions per Origin")
plt.legend(title="Origin", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

# Journal-style font setup
mpl.rcParams.update({
    "font.family": "DejaVu Serif",
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 14,
    "legend.fontsize": 11,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
})

# Compute cumulative percentage (already in df_curves)
df_curves["cumulative_pct"] = (
    df_curves["cumulative_opportunities"] / df_curves["total_possible"] * 100
)

plt.figure(figsize=(10, 7))
colors = plt.cm.tab10.colors

# Plot colored curves per origin
for i, (origin, g) in enumerate(df_curves.groupby("Origin_Name")):
    plt.plot(
        g["car_co2_total"],
        g["cumulative_pct"],
        linewidth=1.5,
        label=origin,
        color=colors[i % len(colors)]
    )

# Highlight cumulative opportunity levels
highlight_levels = [10, 25, 33]
for level in highlight_levels:
    plt.axhline(y=level, color="grey", linestyle="--", alpha=0.7)
    plt.text(
        x=-210,  # slightly left of y-axis
        y=level,
        s=f"{level}%",
        color="grey",
        fontsize=11,
        va="center",
        ha="left",
        backgroundcolor="white"
    )

plt.xlabel("Car CO₂ emissions")
plt.ylabel("Cumulative opportunities (%)")
plt.xlim(0, 10000)
plt.ylim(0, 100)

plt.grid(False)
plt.legend(
    title="Origin",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)

#plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd

highlight_levels = [10, 25, 33]

def find_intersections(group, levels):
    results = []
    x = group["car_co2_total"].values
    y = group["cumulative_pct"].values

    for lvl in levels:
        if y.min() <= lvl <= y.max():
            # interpolate x at y=lvl
            x_at_lvl = np.interp(lvl, y, x)
            results.append((group["Origin_Name"].iloc[0], lvl, x_at_lvl))
    return results

# Build table
intersections = []
for origin, g in df_curves.groupby("Origin_Name"):
    g_sorted = g.sort_values("cumulative_pct")
    intersections.extend(find_intersections(g_sorted, highlight_levels))

df_intersections = pd.DataFrame(intersections, columns=["Origin_Name", "Level_pct", "CO2_value"])

In [ ]:
df_intersections

In [ ]:
# Pivot so each origin has its values across levels
df_wide = df_intersections.pivot(
    index="Origin_Name",
    columns="Level_pct",
    values="CO2_value"
)

# Get City Center row
city_center_vals = df_wide.loc["City Center"]

# Compute ratios
df_ratios = df_wide.divide(city_center_vals)

# Drop City Center itself (always 1x)
df_ratios = df_ratios.drop("City Center")

# Optional: rename columns nicely
df_ratios = df_ratios.rename(columns={10: "10%", 25: "25%", 33: "33%"})
df_ratios.index.name = "Origin"

print(df_ratios.round(2))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ----------------------------
# 1. Standardize CO2 column and add mode
# ----------------------------
df_car_mode  = df_subset_car.rename(columns={"car_co2": "co2_g"}).assign(mode="Car")
df_pt_mode   = df_subset_pt.rename(columns={"pt_co2_total": "co2_g"}).assign(mode="PT")
df_bike_mode = df_subset_bike.rename(columns={"co2_emissions_g": "co2_g"}).assign(mode="Bike")

# ----------------------------
# 2. Combine all modes
# ----------------------------
df_all_modes = pd.concat([df_car_mode, df_pt_mode, df_bike_mode], ignore_index=True)

# ----------------------------
# 3. Compute cumulative opportunities per origin + mode
# ----------------------------
df_curves = (
    df_all_modes
    .sort_values(["Origin_Name", "mode", "co2_g"])
    .groupby(["Origin_Name", "mode"], group_keys=False)
    .apply(lambda g: g.assign(cumulative_opportunities=g["total_opportunities"].cumsum()))
)

# ----------------------------
# 4. Plot
# ----------------------------
plt.figure(figsize=(12,8))

# Colors fixed per origin/hex
palette = sns.color_palette("tab10", n_colors=df_curves["Origin_Name"].nunique())
origin_colors = dict(zip(df_curves["Origin_Name"].unique(), palette))

# Line style per mode
mode_styles = {"Car": "-", "PT": "--", "Bike": ":"}

for (origin, mode), g in df_curves.groupby(["Origin_Name", "mode"]):
    plt.plot(
        g["co2_g"], 
        g["cumulative_opportunities"], 
        color=origin_colors[origin],
        linestyle=mode_styles[mode],
        marker="o",
        label=f"{origin} - {mode}"
    )

plt.xlabel("CO₂ emissions (grams)")
plt.ylabel("Cumulative opportunities reached")
plt.title("Accessibility Curves by Mode and Origin")
plt.grid(True)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()



In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

# ----------------------------
# 1. Standardize CO2 column and add mode
# ----------------------------
df_car_mode  = df_subset_car.rename(columns={"car_co2": "co2_g"}).assign(mode="Car")
df_pt_mode   = df_subset_pt.rename(columns={"pt_co2_total": "co2_g"}).assign(mode="PT")
df_bike_mode = df_subset_bike.rename(columns={"co2_emissions_g": "co2_g"}).assign(mode="Bike")

# ----------------------------
# 2. Combine all modes
# ----------------------------
df_all_modes = pd.concat([df_car_mode, df_pt_mode, df_bike_mode], ignore_index=True)

# ----------------------------
# 3. Compute cumulative opportunities per origin + mode
# ----------------------------
df_curves = (
    df_all_modes
    .sort_values(["Origin_Name", "mode", "co2_g"])
    .groupby(["Origin_Name", "mode"], group_keys=False)
    .apply(lambda g: g.assign(
        cumulative_opportunities=g["total_opportunities"].cumsum(),
        total_possible=g["total_opportunities"].sum()
    ))
    .reset_index(drop=True)
)

# Compute percentage
df_curves["cumulative_pct"] = (
    df_curves["cumulative_opportunities"] / df_curves["total_possible"] * 100
)

# ----------------------------
# 4. Plot
# ----------------------------
mpl.rcParams.update({
    "font.family": "DejaVu Serif",
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 14,
    "legend.fontsize": 11,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
})

plt.figure(figsize=(12, 8))

# Colors fixed per origin
colors = plt.cm.tab10.colors
origin_colors = dict(zip(df_curves["Origin_Name"].unique(), colors))

# Line style per mode
mode_styles = {"Car": "-", "PT": "--", "Bike": ":"}

# Plot per (Origin, Mode)
for (origin, mode), g in df_curves.groupby(["Origin_Name", "mode"]):
    plt.plot(
        g["co2_g"],
        g["cumulative_pct"],
        color=origin_colors[origin],
        linestyle=mode_styles[mode],
        linewidth=1.5,
        label=f"{origin} - {mode}"
    )

# Highlight cumulative levels
highlight_levels = [10, 25, 33]
for level in highlight_levels:
    plt.axhline(y=level, color="grey", linestyle="--", alpha=0.7)
    plt.text(
        x=-0.02*df_curves["co2_g"].max(),
        y=level,
        s=f"{level}%",
        color="grey",
        fontsize=11,
        va="center",
        ha="left",
        backgroundcolor="white"
    )

plt.xlabel("CO₂ emissions (grams)")
plt.ylabel("Cumulative opportunities (%)")
plt.xlim(0, 2400)
plt.ylim(0, 100)

plt.grid(False)
plt.legend(
    title="Origin - Mode",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)

plt.tight_layout()
plt.show()
